<a href="https://colab.research.google.com/github/kaitorr07/Kaito-Raymundo-ML-ITAI1371/blob/main/project_KaitoRaymundo_ITAI1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("https://github.com/kaitorr07/Projects/raw/main/Autoimmune_Disorder_10k_with_All_Disorders.csv")
df['Diagnosis'] = df['Diagnosis'].apply(lambda x: 0 if x == 'Normal' else 1)
df = df.drop('Patient_ID', axis=1)
df = pd.get_dummies(df, columns=['Gender'], drop_first=True)

# Features
features = [
    'Age', 'Sickness_Duration_Months', 'RBC_Count', 'Hemoglobin', 'Hematocrit', 'MCV', 'MCH', 'MCHC', 'RDW', 'Reticulocyte_Count', 'WBC_Count', 'Neutrophils', 'Lymphocytes', 'Monocytes', 'Eosinophils', 'Basophils', 'PLT_Count', 'MPV', 'ANA', 'Esbach', 'MBL_Level', 'ESR', 'C3', 'C4', 'CRP',
    'Anti-dsDNA', 'Anti-Sm', 'Rheumatoid factor', 'ACPA', 'Anti-TPO', 'Anti-Tg', 'Anti-SMA'
    'Low-grade fever', 'Fatigue or chronic tiredness', 'Dizziness', 'Weight loss', 'Rashes and skin lesions', 'Stiffness in the joints', 'Brittle hair or hair loss', 'Dry eyes and/or mouth', 'Joint pain',
    'Anti_dsDNA', 'Anti_enterocyte_antibodies', 'anti_LKM1', 'Anti_RNP', 'ASCA', 'Anti_Ro_SSA', 'Anti_CBir1', 'Anti_BP230', 'DGP', 'Anti_BP180', 'ASMA', 'Anti_IF', 'IgG_IgE_receptor', 'Anti_SRP', 'Anti_desmoglein_3', 'Anti_La_SSB', 'Anti_Jo1', 'ANCA', 'anti_centromere', 'Anti_desmoglein_1', 'EMA', 'Anti_type_VII_collagen', 'C1_inhibitor', 'Anti_TIF1', 'Anti_epidermal_basement_membrane_IgA', 'Anti_OmpC', 'pANCA', 'Anti_tissue_transglutaminase', 'anti_Scl_70', 'Anti_Mi2', 'Anti_parietal_cell', 'Progesterone_antibodies',
    'Gender_Male'
]
target = 'Diagnosis'

existing_features = [f for f in features if f in df.columns]
missing_values_features = df[existing_features].isnull().sum()
missing_values_features = missing_values_features[missing_values_features > 0]

for col in missing_values_features.index:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)

# ---Model-specific code---

# Separating features and target
X = df[existing_features]
y = df[target]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Training
model = LogisticRegression(max_iter=10000, solver='liblinear', C=1e-6)
model.fit(X_train, y_train)

#Sort Option 1: Sickness Duration
df_sickness_duration_sorted = df.sort_values(by='Sickness_Duration_Months', ascending=True)
print("Top 20 'most treatable' cases based on Sickness Duration:")
display(df_sickness_duration_sorted[['Diagnosis', 'Sickness_Duration_Months', 'Age', 'Gender_Male']].head(20))

#Sort Option 2: Indication of Autoimmunity (Lab Results)
severity_markers = ['ESR', 'CRP', 'Anti-dsDNA', 'Anti-Sm', 'Rheumatoid factor', 'ACPA']
existing_severity_markers = [m for m in severity_markers if m in df.columns]
df['severity_score'] = df[existing_severity_markers].sum(axis=1)
df_severity_lab_results_sorted = df.sort_values(by='severity_score', ascending=True)
print("Top 20 'most treatable' cases based on Indication of Autoimmunity from Lab Results (lower severity score = higher treatability)")
display(df_severity_lab_results_sorted[['Patient ID', 'Diagnosis', 'severity_score', 'Age', 'Gender_Male']].head(20))

# ---Results/Testing---

# Prediction/Accuracy
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print('Classification Report (from single train/test split):\n')
print(classification_report(y_test, y_pred))
print('Accuracy (from single train/test split): ', accuracy)

# Cross Validation
print('\nCross-Validation:\n')

cv_model = LogisticRegression(max_iter=10000, solver='liblinear', C=1e-6)
cv_scores = cross_val_score(cv_model, X, y, cv=39, scoring='accuracy')

print('Cross-validation accuracies:', cv_scores)
print('\nMean cross-validation accuracy: %0.2f (+/- %0.2f)' % (cv_scores.mean(), cv_scores.std() * 2))

Top 20 'most treatable' cases based on Sickness Duration:


,Diagnosis,Sickness_Duration_Months,Age,Gender_Male
12473,0,0,31,False
12474,0,0,71,True
12475,0,0,83,False
12476,0,0,70,True
12477,0,0,61,False
12478,0,0,52,False
12479,0,0,27,False
12480,0,0,76,True
12481,0,0,65,False
12482,0,0,80,False


Top 20 'most treatable' cases based on Indication of Autoimmunity from Lab Results (lower severity score = higher treatability)


KeyError: "['Patient ID'] not in index"